In [1]:
!pip install rdflib SPARQLWrapper

In [3]:
# build initial RDF graph from extracted entities and relations
import csv
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL, XSD
from pathlib import Path

# Namespaces
CC = Namespace("http://climate-change.org/kg/")
WD = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")

g = Graph()
g.bind("cc", CC)
g.bind("wd", WD)
g.bind("wdt", WDT)
g.bind("owl", OWL)
g.bind("rdfs", RDFS)

# ontology classes
PERSON = CC.Person
ORG = CC.Organization
GPE = CC.GeopoliticalEntity
DATE = CC.Date

label_to_class = {
    "PERSON": PERSON,
    "ORG": ORG,
    "GPE": GPE,
    "DATE": DATE,
}

def make_uri(text):
    cleaned = text.strip().replace(" ", "_").replace("/", "-").replace(".", "").replace(",", "")
    return CC[cleaned]

in_path = Path("extracted_knowledge.csv")
entities = {}

with open(in_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        text = row["entity_text"].strip()
        label = row["entity_label"].strip()
        if label not in label_to_class:
            continue
        uri = make_uri(text)
        if uri not in entities:
            entities[uri] = (text, label)
            cls = label_to_class[label]
            g.add((uri, RDF.type, cls))
            g.add((uri, RDFS.label, Literal(text, lang="en")))

print(f"entities added: {len(entities)}")

# Ajout des relations candidates
REL_PATH = Path("candidate_relations.csv")

relations_added = 0
with open(REL_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        subj_text = row["subject"].strip()
        obj_text = row["object"].strip()
        rel_text = row["relation"].strip()
        subj_uri = make_uri(subj_text)
        obj_uri = make_uri(obj_text)
        rel_uri = CC[rel_text]
        if (subj_uri, RDF.type, None) in g and (obj_uri, RDF.type, None) in g:
            g.add((subj_uri, rel_uri, obj_uri))
            relations_added += 1

print(f"relations added: {relations_added}")
print(f"total triples: {len(g)}")

out_path = Path("climate_kg.ttl")
g.serialize(destination=str(out_path), format="turtle")
print(f"graph saved: {out_path.resolve()}")

entities added: 212
relations added: 1199
total triples: 1589
graph saved: C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\climate_kg.ttl


In [7]:
# owl ontology definition
onto = Graph()
onto.bind("cc", CC)
onto.bind("owl", OWL)
onto.bind("rdfs", RDFS)
onto.bind("xsd", XSD)

# declare ontology
onto_uri = URIRef("http://climate-change.org/kg/ontology")
onto.add((onto_uri, RDF.type, OWL.Ontology))
onto.add((onto_uri, RDFS.label, Literal("Climate Change Knowledge Graph Ontology", lang="en")))

# classes
classes = [
    (CC.Person, "Person", "A human individual involved in climate change research or policy."),
    (CC.Organization, "Organization", "An institution, agency, or body related to climate change."),
    (CC.GeopoliticalEntity, "GeopoliticalEntity", "A country, region, or political territory."),
    (CC.Date, "Date", "A concrete temporal reference such as a year or dated event."),
    (CC.ClimateEvent, "ClimateEvent", "A climate-related phenomenon or event such as a heatwave or flood."),
    (CC.Agreement, "Agreement", "An international treaty or agreement related to climate policy."),
]

for uri, label, comment in classes:
    onto.add((uri, RDF.type, OWL.Class))
    onto.add((uri, RDFS.label, Literal(label, lang="en")))
    onto.add((uri, RDFS.comment, Literal(comment, lang="en")))

#sub classes
onto.add((CC.Agreement, RDFS.subClassOf, CC.Organization))
onto.add((CC.ClimateEvent, RDFS.subClassOf, OWL.Thing))

#object prop
object_properties = [
    (CC.affiliatedWith, "affiliatedWith", CC.Person, CC.Organization),
    (CC.locatedIn, "locatedIn", CC.Organization, CC.GeopoliticalEntity),
    (CC.signedBy, "signedBy", CC.Agreement, CC.GeopoliticalEntity),
    (CC.occursIn, "occursIn", CC.ClimateEvent, CC.GeopoliticalEntity),
]

for uri, label, domain, range_ in object_properties:
    onto.add((uri, RDF.type, OWL.ObjectProperty))
    onto.add((uri, RDFS.label, Literal(label, lang="en")))
    onto.add((uri, RDFS.domain, domain))
    onto.add((uri, RDFS.range, range_))

#data prop
onto.add((CC.hasYear, RDF.type, OWL.DatatypeProperty))
onto.add((CC.hasYear, RDFS.label, Literal("hasYear", lang="en")))
onto.add((CC.hasYear, RDFS.domain, CC.Date))
onto.add((CC.hasYear, RDFS.range, XSD.integer))

onto.add((CC.confidenceScore, RDF.type, OWL.DatatypeProperty))
onto.add((CC.confidenceScore, RDFS.label, Literal("confidenceScore", lang="en")))
onto.add((CC.confidenceScore, RDFS.range, XSD.float))

ONTO_PATH = Path("ontology.ttl")
onto.serialize(destination=str(ONTO_PATH), format="turtle")
print(f"ontology saved: {ONTO_PATH.resolve()}")
print(f"ontology triples: {len(onto)}")

ontology saved: C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\ontology.ttl
ontology triples: 45


In [15]:
# manual alignements
# QIDs verified on wikidata.org

manual_alignments = [
    # (entity text, wikidata QID, confidence score)
    ("IPCC",                                          "Q42262",   1.0),
    ("Intergovernmental Panel on Climate Change",     "Q42262",   1.0),
    ("the Intergovernmental Panel on Climate Change", "Q42262",   1.0),
    ("The Intergovernmental Panel on Climate Change", "Q42262",   1.0),
    ("The UN Intergovernmental Panel on Climate Change", "Q42262","1.0"),
    ("United Nations",                                "Q1065",    1.0),
    ("UN",                                            "Q1065",    0.9),
    ("United Nations Framework Convention on Climate Change", "Q170322", 1.0),
    ("UNFCCC",                                        "Q170322",  1.0),
    ("the UN Environment Programme",                  "Q160463",  1.0),
    ("United Nations Environment",                    "Q160463",  0.9),
    ("the World Meteorological Organization",         "Q167691",  1.0),
    ("WMO",                                           "Q167691",  0.9),
    ("NASA",                                          "Q23548",   1.0),
    ("NOAA",                                          "Q48268",   1.0),
    ("FEMA",                                          "Q273296",  1.0),
    ("Cambridge University Press",                    "Q912350",  1.0),
    ("Al Gore",                                       "Q19673",   1.0),
    ("United States",                                 "Q30",      1.0),
    ("the United States",                             "Q30",      0.9),
    ("U.S.",                                          "Q30",      0.9),
    ("New York",                                      "Q60",      0.9),
    ("Paris",                                         "Q90",      0.9),
    ("Cambridge",                                     "Q350",     0.8),
    ("United Kingdom",                                "Q145",     1.0),
    ("UK",                                            "Q145",     0.9),
    ("the U.S. Global Change Research Program",       "Q7887823", 1.0),
    ("Conference of the Parties",                     "Q1761307", 0.9),
]

alignment_graph = Graph()
alignment_graph.bind("cc", CC)
alignment_graph.bind("wd", WD)
alignment_graph.bind("owl", OWL)
alignment_graph.bind("rdfs", RDFS)
alignment_graph.bind("xsd", XSD)

mapping_table = []

for text, qid, confidence in manual_alignments:
    uri = make_uri(text)
    wd_uri = WD[qid]
    alignment_graph.add((uri, OWL.sameAs, wd_uri))
    alignment_graph.add((uri, CC.confidenceScore, Literal(float(confidence), datatype=XSD.float)))
    mapping_table.append((text, str(wd_uri), float(confidence)))

print(f"aligned: {len(mapping_table)} entities")
print(f"\n{'Entity':<55} {'Wikidata URI':<35} {'Score'}")
print("-" * 95)
for text, wd_uri, score in mapping_table:
    print(f"{text:<55} {wd_uri:<35} {score:.2f}")

ALIGN_PATH = Path("alignment.ttl")
alignment_graph.serialize(destination=str(ALIGN_PATH), format="turtle")
print(f"\nalignment saved: {ALIGN_PATH.resolve()}")
print(f"alignment triples: {len(alignment_graph)}")

aligned: 28 entities

Entity                                                  Wikidata URI                        Score
-----------------------------------------------------------------------------------------------
IPCC                                                    http://www.wikidata.org/entity/Q42262 1.00
Intergovernmental Panel on Climate Change               http://www.wikidata.org/entity/Q42262 1.00
the Intergovernmental Panel on Climate Change           http://www.wikidata.org/entity/Q42262 1.00
The Intergovernmental Panel on Climate Change           http://www.wikidata.org/entity/Q42262 1.00
The UN Intergovernmental Panel on Climate Change        http://www.wikidata.org/entity/Q42262 1.00
United Nations                                          http://www.wikidata.org/entity/Q1065 1.00
UN                                                      http://www.wikidata.org/entity/Q1065 0.90
United Nations Framework Convention on Climate Change   http://www.wikidata.org/entity/Q17032

In [17]:
from SPARQLWrapper import SPARQLWrapper, JSON

def test_sparql():
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    sparql.setQuery("""
        SELECT ?label WHERE {
            wd:Q42262 rdfs:label ?label .
            FILTER(LANG(?label) = "en")
        }
        LIMIT 1
    """)
    sparql.setReturnFormat(JSON)
    sparql.addCustomHttpHeader("User-Agent", "ClimateKG/1.0 (student project)")
    try:
        results = sparql.query().convert()
        print("SPARQL OK:", results["results"]["bindings"][0]["label"]["value"])
    except Exception as e:
        print("SPARQL FAILED:", e)

test_sparql()

SPARQL OK: European Space Agency


In [19]:
from SPARQLWrapper import SPARQLWrapper, JSON
import time

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)
sparql.addCustomHttpHeader("User-Agent", "ClimateKG/1.0 (student project)")

# Core QIDs to expand from (our best-aligned entities)
core_qids = [
    "Q42262",   # IPCC
    "Q1065",    # United Nations
    "Q170322",  # UNFCCC
    "Q160463",  # UN Environment Programme
    "Q167691",  # WMO
    "Q23548",   # NASA
    "Q48268",   # NOAA
    "Q19673",   # Al Gore
    "Q30",      # United States
    "Q145",     # United Kingdom
    "Q90",      # Paris
    "Q1761307", # Conference of the Parties
    "Q7887823", # U.S. Global Change Research Program
]

def expand_entity(qid, limit=500):
    """1-hop expansion: fetch all (qid, p, o) triples from Wikidata."""
    query = f"""
        SELECT ?p ?o WHERE {{
            wd:{qid} ?p ?o .
            FILTER(!isLiteral(?o) || LANG(?o) = "en" || LANG(?o) = "")
        }}
        LIMIT {limit}
    """
    sparql.setQuery(query)
    try:
        results = sparql.query().convert()
        triples = []
        for row in results["results"]["bindings"]:
            p = row["p"]["value"]
            o = row["o"]["value"]
            triples.append((qid, p, o))
        return triples
    except Exception as e:
        print(f"  Error on {qid}: {e}")
        return []

expanded_graph = Graph()
expanded_graph.bind("cc", CC)
expanded_graph.bind("wd", WD)
expanded_graph.bind("wdt", WDT)
expanded_graph.bind("owl", OWL)
expanded_graph.bind("rdfs", RDFS)

# Merge initial graph and alignment into expanded graph
for triple in g:
    expanded_graph.add(triple)
for triple in onto:
    expanded_graph.add(triple)
for triple in alignment_graph:
    expanded_graph.add(triple)

print(f"Starting triples (initial + ontology + alignment): {len(expanded_graph)}")

total_added = 0
for qid in core_qids:
    triples = expand_entity(qid, limit=500)
    count = 0
    for _, p, o in triples:
        s_uri = WD[qid]
        p_uri = URIRef(p)
        o_uri = URIRef(o) if o.startswith("http") else Literal(o)
        if (s_uri, p_uri, o_uri) not in expanded_graph:
            expanded_graph.add((s_uri, p_uri, o_uri))
            count += 1
    total_added += count
    print(f"  {qid}: +{count} triples")
    time.sleep(1)

print(f"\nTotal new triples added: {total_added}")
print(f"Total triples in expanded graph: {len(expanded_graph)}")

Starting triples (initial + ontology + alignment): 1690
  Q42262: +417 triples
  Q1065: +500 triples
  Q170322: +296 triples
  Q160463: +9 triples
  Q167691: +58 triples
  Q23548: +500 triples
  Q48268: +270 triples
  Q19673: +500 triples
  Q30: +500 triples
  Q145: +500 triples
  Q90: +500 triples
  Q1761307: +0 triples
  Q7887823: +15 triples

Total new triples added: 4065
Total triples in expanded graph: 5755


In [21]:
def get_linked_entities(qid, limit=20):
    """Fetch URIs linked from a QID (objects that are Wikidata entities)."""
    query = f"""
        SELECT DISTINCT ?o WHERE {{
            wd:{qid} ?p ?o .
            FILTER(STRSTARTS(STR(?o), "http://www.wikidata.org/entity/Q"))
        }}
        LIMIT {limit}
    """
    sparql.setQuery(query)
    try:
        results = sparql.query().convert()
        return [row["o"]["value"].split("/")[-1]
                for row in results["results"]["bindings"]]
    except Exception:
        return []

# Collect 2-hop neighbors from core entities
second_hop_qids = set()
for qid in core_qids:
    neighbors = get_linked_entities(qid, limit=20)
    second_hop_qids.update(neighbors)
    time.sleep(0.5)

# Remove QIDs already in core
second_hop_qids -= set(core_qids)
print(f"2-hop entities to expand: {len(second_hop_qids)}")

# Expand each 2-hop entity (smaller limit to stay within 200k)
total_added_2hop = 0
for qid in list(second_hop_qids):
    triples = expand_entity(qid, limit=200)
    count = 0
    for _, p, o in triples:
        s_uri = WD[qid]
        p_uri = URIRef(p)
        o_uri = URIRef(o) if o.startswith("http") else Literal(o)
        if (s_uri, p_uri, o_uri) not in expanded_graph:
            expanded_graph.add((s_uri, p_uri, o_uri))
            count += 1
    total_added_2hop += count
    time.sleep(0.3)

print(f"2-hop triples added: {total_added_2hop}")
print(f"Total triples after 2-hop: {len(expanded_graph)}")

2-hop entities to expand: 187
2-hop triples added: 22037
Total triples after 2-hop: 27792


In [23]:
# Remove duplicate triples (rdflib handles this natively)
# Remove triples with empty literals
to_remove = []
for s, p, o in expanded_graph:
    if isinstance(o, Literal) and str(o).strip() == "":
        to_remove.append((s, p, o))

for triple in to_remove:
    expanded_graph.remove(triple)

print(f"Removed {len(to_remove)} empty literal triples")
print(f"Final triple count: {len(expanded_graph)}")

# Count entities and relations
entities_count = len(set(s for s, p, o in expanded_graph))
relations_count = len(set(p for s, p, o in expanded_graph))
print(f"Entities (subjects): {entities_count}")
print(f"Distinct relations: {relations_count}")

# Export
expanded_path = Path("expanded_kb.nt")
expanded_graph.serialize(destination=str(expanded_path), format="nt")
print(f"\nExpanded KB saved: {expanded_path.resolve()}")

Removed 0 empty literal triples
Final triple count: 27792
Entities (subjects): 424
Distinct relations: 2165


C:\Users\rache\anaconda3\Lib\site-packages\rdflib\plugins\serializers\nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(



Expanded KB saved: C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\expanded_kb.nt


In [27]:
# Climate Change specific QIDs for additional expansion
climate_qids = [
    "Q167639",  # Paris Agreement
    "Q181600",  # Kyoto Protocol
    "Q131204",  # climate change
    "Q3397883", # global warming
    "Q484083",  # greenhouse gas
    "Q1827102", # carbon dioxide emissions
    "Q179731",  # fossil fuel
    "Q4697582", # sea level rise
    "Q208685",  # Arctic ice
    "Q181014",  # renewable energy
    "Q80151",   # solar energy
    "Q11457",   # wind power
    "Q178694",  # deforestation
    "Q208042",  # biodiversity loss
    "Q8667",    # flood
    "Q8060",    # drought
    "Q134783",  # wildfire
    "Q193034",  # heat wave
    "Q83303",   # IPCC Assessment Report
    "Q3041682", # carbon tax
    "Q189819",  # cap and trade
    "Q7174",    # World Bank
    "Q8908",    # European Union
    "Q258691",  # COP21
    "Q860580",  # COP26
    "Q15156637",# COP28
]

total_added_climate = 0
for qid in climate_qids:
    triples = expand_entity(qid, limit=500)
    count = 0
    for _, p, o in triples:
        s_uri = WD[qid]
        p_uri = URIRef(p)
        o_uri = URIRef(o) if o.startswith("http") else Literal(o)
        if (s_uri, p_uri, o_uri) not in expanded_graph:
            expanded_graph.add((s_uri, p_uri, o_uri))
            count += 1
    total_added_climate += count
    print(f"  {qid}: +{count} triples")
    time.sleep(1)

print(f"\nClimate triples added: {total_added_climate}")
print(f"Total triples: {len(expanded_graph)}")

  Q167639: +130 triples
  Q181600: +171 triples
  Q131204: +52 triples
  Error on Q3397883: HTTP Error 502: Bad Gateway
  Q3397883: +0 triples
  Q484083: +138 triples
  Q1827102: +46 triples
  Q179731: +286 triples
  Q4697582: +52 triples
  Q208685: +297 triples
  Q181014: +152 triples
  Q80151: +185 triples
  Q11457: +262 triples
  Q178694: +140 triples
  Q208042: +150 triples
  Q8667: +169 triples
  Q8060: +283 triples
  Q134783: +131 triples
  Q193034: +94 triples
  Q83303: +226 triples
  Q3041682: +30 triples
  Q189819: +132 triples
  Q7174: +332 triples
  Q8908: +244 triples
  Q258691: +50 triples
  Q860580: +350 triples
  Q15156637: +0 triples

Climate triples added: 4102
Total triples: 31894


In [29]:
# Final cleanup
to_remove = []
for s, p, o in expanded_graph:
    if isinstance(o, Literal) and str(o).strip() == "":
        to_remove.append((s, p, o))
for triple in to_remove:
    expanded_graph.remove(triple)

total = len(expanded_graph)
entities_count = len(set(s for s, p, o in expanded_graph))
relations_count = len(set(p for s, p, o in expanded_graph))

print(f"Final triple count:     {total}")
print(f"Entities (subjects):    {entities_count}")
print(f"Distinct relations:     {relations_count}")
print(f"Target range:           50,000 - 200,000 triples")
print(f"Status: {'OK' if 50000 <= total <= 200000 else 'BELOW TARGET' if total < 50000 else 'ABOVE TARGET'}")

expanded_path = Path("expanded_kb.nt")
expanded_graph.serialize(destination=str(expanded_path), format="nt")
print(f"\nExpanded KB saved: {expanded_path.resolve()}")

# KB statistics file
STATS_PATH = Path("kb_statistics.txt")
with open(STATS_PATH, "w", encoding="utf-8") as f:
    f.write("KB Statistics\n")
    f.write("=============\n\n")
    f.write(f"Total triples:       {total}\n")
    f.write(f"Entities (subjects): {entities_count}\n")
    f.write(f"Distinct relations:  {relations_count}\n")
    f.write("\nSources\n")
    f.write("-------\n")
    f.write("Initial graph (from crawler + NER):  climate_kg.ttl\n")
    f.write("Ontology:                            ontology.ttl\n")
    f.write("Alignment (28 entities, Wikidata):   alignment.ttl\n")
    f.write("Expanded KB (1-hop + 2-hop + thematic expansion): expanded_kb.nt\n")

print(f"Statistics saved: {STATS_PATH.resolve()}")

Final triple count:     31894
Entities (subjects):    447
Distinct relations:     2658
Target range:           50,000 - 200,000 triples
Status: BELOW TARGET

Expanded KB saved: C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\expanded_kb.nt
Statistics saved: C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\kb_statistics.txt


In [31]:
# Collect neighbors of the climate QIDs we just added
climate_neighbors = set()
for qid in climate_qids:
    neighbors = get_linked_entities(qid, limit=30)
    climate_neighbors.update(neighbors)
    time.sleep(0.3)

# Remove already expanded QIDs
already_done = set(core_qids) | set(climate_qids) | second_hop_qids
climate_neighbors -= already_done
print(f"New neighbors to expand: {len(climate_neighbors)}")

total_added_neighbors = 0
for qid in list(climate_neighbors):
    triples = expand_entity(qid, limit=300)
    count = 0
    for _, p, o in triples:
        s_uri = WD[qid]
        p_uri = URIRef(p)
        o_uri = URIRef(o) if o.startswith("http") else Literal(o)
        if (s_uri, p_uri, o_uri) not in expanded_graph:
            expanded_graph.add((s_uri, p_uri, o_uri))
            count += 1
    total_added_neighbors += count
    time.sleep(0.3)

print(f"Neighbor triples added: {total_added_neighbors}")
print(f"Total triples: {len(expanded_graph)}")

New neighbors to expand: 393
  Error on Q1516211: HTTP Error 502: Bad Gateway
Neighbor triples added: 38676
Total triples: 70570


In [33]:
to_remove = []
for s, p, o in expanded_graph:
    if isinstance(o, Literal) and str(o).strip() == "":
        to_remove.append((s, p, o))
for triple in to_remove:
    expanded_graph.remove(triple)

total = len(expanded_graph)
entities_count = len(set(s for s, p, o in expanded_graph))
relations_count = len(set(p for s, p, o in expanded_graph))

print(f"Final triple count:     {total}")
print(f"Entities (subjects):    {entities_count}")
print(f"Distinct relations:     {relations_count}")
print(f"Status: {'OK' if 50000 <= total <= 200000 else 'BELOW TARGET' if total < 50000 else 'ABOVE TARGET'}")

expanded_path = Path("expanded_kb.nt")
expanded_graph.serialize(destination=str(expanded_path), format="nt")
print(f"Expanded KB saved: {expanded_path.resolve()}")

STATS_PATH = Path("kb_statistics.txt")
with open(STATS_PATH, "w", encoding="utf-8") as f:
    f.write("KB Statistics\n")
    f.write("=============\n\n")
    f.write(f"Total triples:       {total}\n")
    f.write(f"Entities (subjects): {entities_count}\n")
    f.write(f"Distinct relations:  {relations_count}\n")
    f.write("\nSources\n")
    f.write("-------\n")
    f.write("Initial graph (from crawler + NER):                      climate_kg.ttl\n")
    f.write("Ontology:                                                 ontology.ttl\n")
    f.write("Alignment (28 entities, Wikidata):                        alignment.ttl\n")
    f.write("Expanded KB (1-hop + 2-hop + thematic + neighbor):        expanded_kb.nt\n")
print(f"Statistics saved: {STATS_PATH.resolve()}")

Final triple count:     70570
Entities (subjects):    839
Distinct relations:     4205
Status: OK
Expanded KB saved: C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\expanded_kb.nt
Statistics saved: C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\kb_statistics.txt
